In [1]:
import os

In [2]:
%pwd

'd:\\projects\\project-9 kidney disease\\kidney-desease-classification_again\\kidney-classification\\research'

In [3]:
os.chdir("../")

%pwd

In [5]:
%pwd

'd:\\projects\\project-9 kidney disease\\kidney-desease-classification_again\\kidney-classification'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzipped_data_dir: Path
    

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
        def get_data_ingestion_config(self) -> DataIngestionConfig:
            config = self.config.data_ingestion
            
            create_directories([config.root_dir])
            
            data_ingestion_config = DataIngestionConfig(
                root_dir=config.root_dir,
                source_URL=config.source_URL,
                local_data_file=config.local_data_file,
                unzipped_data_dir=config.unzipped_data_dir
            )
            
            return data_ingestion_config

In [12]:
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [14]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self)-> str:
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs(self.config.root_dir, exist_ok = True)
            logger.info(f"downloading data from {dataset_url} into file {zip_download_dir}")
            
            file_id = dataset_url.split('/')[-2]
            gdown.download(id=file_id, output=zip_download_dir)
            size = get_size(Path(zip_download_dir))
            logger.info(f"file downloaded successfully and saved at {zip_download_dir} with size {size}")
            return zip_download_dir
        except Exception as e:
            raise e
    
    
    def extract_zip_file(self, zip_file_path: str):
        try:
            logger.info(f"extraction of file: {zip_file_path} started")
            unzip_dir = self.config.unzipped_data_dir
            os.makedirs(unzip_dir, exist_ok = True)
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_dir)
            logger.info(f"extraction completed successfully at location: {unzip_dir}")
        except Exception as e:
            raise e

In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    
    data_ingestion = DataIngestion(config = data_ingestion_config)
    zip_file_path = data_ingestion.download_file()
    data_ingestion.extract_zip_file(zip_file_path=zip_file_path)
except Exception as e:
    raise e